# Longest Common Token Subsequence: Update 2025-04-24

This algorithm is developed and designed to compare two text sequences, especially of person/company/region name: e.g. 'Alan Turing' vs. 'Alan Mathison Turing', 'United States of America' vs. 'U.S.A.', etc.

The code is based on the Longest Common Subsequence algorithm, where the order of characters/tokens/words matters and therefore a lower matching score is potentially calculated for some cases: e.g. 'Albert Einstein' vs. 'Einstein, Albert'. To overcome this challenge, the code locally performs the Longest Common Subsequence algorithm on every possible pair of tokens, whose time complexity still remains O(mn), where m = len(text1) and n = len(text2). 

And then, an algorithm for finding the maximum sum of values from non-overlappding rectangles is used to find the best matching result. Here, I am currently using a two-step approach: (1) find all the possible groups of non-ovelapping rectangles in terms of x-axis, (2) for each group, narrown down non-overlapping rectangles, in terms of y-axis as well, which gives the maximum sum of values within this group, and finally (3) find the groups that give the maximum sum of values across all the groups found in (1) and (2). This may not be an ideally optimized method but can find all possible groups of non-overlapping rectangles over both x-axis and y-axis. 

Also, this algorithm may not be really ideal for comparison of general long text.

Please see the test result below on some examples. 

Bomsoo Kim

- Utils

In [1]:
def is_word_char(c, word_char_set=set('0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ')):
    return c in word_char_set

if __name__=='__main__':
    print(is_word_char('c'))
    print(is_word_char('-'))
    print(is_word_char('#'))

True
False
False


In [2]:
def find_tokens(text):
    #--- forward cumulutive sum of characters -----------------------
    csum = [0] * len(text) # initialize to zeros
    for i in range(len(text)-1, -1, -1):
        if is_word_char(text[i]):
            if i == len(text)-1: # if the very last charcter
                csum[i] = 1
            else:
                csum[i] = csum[i+1] + 1

    #--- token connectivity = root ----------------------------------
    root = [None] * len(text)
    iroot, cnt = {}, 0
    for i in range(len(text)):
        if is_word_char(text[i]):
            if i == 0 or not is_word_char(text[i-1]): # if the initial character of each token
                i_initial = i
                iroot[i], cnt = cnt, cnt + 1
            root[i] = i_initial

    return csum, root, iroot

if __name__=="__main__":
    print(find_tokens('hello world!'))
    # ([5, 4, 3, 2, 1, 0, 5, 4, 3, 2, 1, 0], [0, 0, 0, 0, 0, None, 6, 6, 6, 6, 6, None], {0: 0, 6: 1})

([5, 4, 3, 2, 1, 0, 5, 4, 3, 2, 1, 0], [0, 0, 0, 0, 0, None, 6, 6, 6, 6, 6, None], {0: 0, 6: 1})


In [3]:
def flatten_trie_node(node, arr, matches):
    if not node:
        matches.append(list(arr))
        return
    
    for k in node.keys():
        arr.append(k)
        flatten_trie_node(node[k], arr, matches)
        arr.pop()
    return

In [4]:
def log_matched_results(text1, text2, match1, match2):
    match_out1 = f"{text1}\n{''.join(['^' if n >=0 else ' ' for n in match1])}\n{''.join([chr(ord('A')+n) if n >=0 else ' ' for n in match1])}"
    match_out2 = f"{text2}\n{''.join(['^' if n >=0 else ' ' for n in match2])}\n{''.join([chr(ord('A')+n) if n >=0 else ' ' for n in match2])}"
    log = f'{match_out1}\n{match_out2}'
    return log

- Maximum value of non-overlapping intervals with all possible solutions
- [1235. Maximum Profit in Job Scheduling](https://leetcode.com/problems/maximum-profit-in-job-scheduling/description/)
- [2008. Maximum Earnings From Taxi](https://leetcode.com/problems/maximum-earnings-from-taxi/description/)
- [1751. Maximum Number of Events That Can Be Attended II](https://leetcode.com/problems/maximum-number-of-events-that-can-be-attended-ii/description/)

In [5]:
def get_all_non_overlapping_intervals(i, intervals, START=0, END=1, VALUE=2, dp=None, get_trie=True): # tested 2025-04-25
    if i in dp:
        return dp[i]

    if i >= len(intervals):
        return 0, {}
    
    #--- binary search --------------------------
    x, y = i, len(intervals)
    while x + 1 < y:
        m = (x + y) // 2
        if intervals[i][END] < intervals[m][START]: # <, when [1,2] and [3,4] is considered non-overlapping
        # if intervals[i][END] <= intervals[m][START]: # <=, when [1,2] and [2,3] is considered non-overlapping
            y = m
        else:
            x = m

    out_ = get_all_non_overlapping_intervals(x + 1, intervals, START=START, END=END, VALUE=VALUE, dp=dp, get_trie=get_trie)
    out1 = intervals[i][VALUE] + out_[0], {i: out_[1]}
    out2 = get_all_non_overlapping_intervals(i + 1, intervals, START=START, END=END, VALUE=VALUE, dp=dp, get_trie=get_trie)

    max_val = max(out1[0], out2[0])
    trie = {}
    if get_trie:
        if out1[0] == max_val:
            trie.update(out1[1])
        if out2[0] == max_val:
            trie.update(out2[1])

    dp[i] = max_val, trie
    return dp[i]

if __name__=='__main__':
    # intervals = [[4,6,3],[2,5,7]]
    intervals = sorted([[1,3,3],[4,6,3],[2,5,6]])

    max_val, trie = get_all_non_overlapping_intervals(0, intervals, dp={})

    matches = []
    flatten_trie_node(trie, [], matches)
    print(matches)

[[0, 2], [1]]


- Longest common subsequence, with all possible solutions
- [1143. Longest Common Subsequence](https://leetcode.com/problems/longest-common-subsequence/description/)

In [6]:
def lcs(i, j, text1, text2, dp=None, get_trie=True): # tested 2025-04-25
    if (i,j) in dp:
        return dp[(i,j)]
   
    if i >= len(text1) or j >= len(text2):
        return 0, {}
    
    if text1[i] == text2[j]:       
        out_ = lcs(i+1, j+1, text1, text2, dp=dp, get_trie=get_trie)
        out0 = 1 + out_[0], {(i,j):out_[1]}
    else:
        out0 = -1, {}
    out1 = lcs(i, j+1, text1, text2, dp=dp, get_trie=get_trie)
    out2 = lcs(i+1, j, text1, text2, dp=dp, get_trie=get_trie)
    
    max_val = max(out0[0], out1[0], out2[0])
    trie = {}
    if get_trie:
        if out0[0] == max_val:
            trie.update(out0[1])
        if out1[0] == max_val:
            trie.update(out1[1])
        if out2[0] == max_val:
            trie.update(out2[1])
    
    dp[(i,j)] = max_val, trie
    return dp[(i,j)]


- Longest common token subsequence (~ Brute force)

In [7]:
def longest_common_token_subsequence(text1, text2):
    all_matches = []

    csum1, root1, iroot1 = find_tokens(text1)
    csum2, root2, iroot2 = find_tokens(text2)

    #--- token-by-token longest common subsequence: greedy for each token-token pair --------------------------------------
    for i in iroot1.keys():
        for j in iroot2.keys():
            _text1_ = text1[i:(i + csum1[i])]
            _text2_ = text2[j:(j + csum2[j])]

            val, trie = lcs(0, 0, _text1_, _text2_, dp={})

            matches = []
            flatten_trie_node(trie, [], matches)
            # print(f'_text1_ = {_text1_}, _text2_ = {_text2_}, matches = {matches}')

            for pairs in matches:
                if pairs:
                    all_matches.append([(i+x, j+y) for x,y in pairs])

    #--- sort along x-axis and find non-overlapping rectanlges over x-axis ------------------------------------------------
    #--- there can be still some recangles overlapping over y-axis, but they will be removed in the next step -------------
    AXIS = 0 # x
    SCORE = 0
    intervals_0 = [[pairs[0][AXIS], pairs[-1][AXIS], SCORE, k] for k, pairs in enumerate(all_matches)] # SCORE = 0, to find all possible group of non-overlapping rectangles
    intervals_0 = sorted(intervals_0)

    _, trie_0 = get_all_non_overlapping_intervals(0, intervals_0, dp={})
    matches_0 = []
    flatten_trie_node(trie_0, [], matches_0)
    # print(matches_0)

    #--- sort along y-axis and find non-overlapping rectangles over y-axis ------------------------------------------------
    ans = []
    seen = set()
    for mm in matches_0:
        mm_orig_0 = [intervals_0[i][3] for i in mm]
        # print(mm, mm_orig_0)

        AXIS = 1 # y
        intervals_1 = [[all_matches[k][0][AXIS], all_matches[k][-1][AXIS], len(all_matches[k]), k] for k in mm_orig_0] # SCORE = len(all_matches[k])
        intervals_1 = sorted(intervals_1)

        _, trie_1 = get_all_non_overlapping_intervals(0, intervals_1, dp={})
        matches_1 = []
        flatten_trie_node(trie_1, [], matches_1)

        for mm_1 in matches_1:
            mm_orig_1 = [intervals_1[i][3] for i in mm_1]

            key_seen = tuple(sorted(mm_orig_1))
            if key_seen not in seen:
                score = sum(len(all_matches[i]) for i in mm_orig_1)
                
                while ans and ans[-1][0] < score:
                    _ = ans.pop()
                
                if (not ans) or (ans[-1][0] == score):
                    ans.append((score, mm_orig_1))

                seen.add(key_seen)

    #--- find matched results ------------------------------------------------------------------
    outs = []
    for _, option in ans:
        match1, match2 = [-1]*len(text1), [-1]*len(text2) # initialize
        for k, i in enumerate(option):
            pairs = all_matches[i]
            for x, y in pairs:
                if match1[x] < 0:
                    match1[x] = k
                else:
                    raise(Exception('Brad error: a critical issue occurred. There is an overlap...'))
                
                if match2[y] < 0:
                    match2[y] = k
                else:
                    raise(Exception('Brad error: a critical issue occurred. There is an overlap...'))

        outs.append({'match1':match1, 'match2':match2,})

    out = {'csum1':csum1, 'root1':root1, 'iroot1':iroot1, 'csum2':csum2, 'root2':root2, 'iroot2':iroot2, 'matches': outs}

    return out

# if __name__=='__main__':
    # # text1 = 'xab'
    # # text2 = 'axb'
    # # text1 = 'xxoxxx'
    # # text2 = 'xxx'
    # # text1 = 'xxxxx'
    # # text2 = 'oxxx'
    # # text1 = 'oxxxxx'
    # # text2 = 'xxx'
    # # text1 = 'xxccxab'
    # # text2 = 'axb'
    # # text1 = 'xxxxxxyxxx'
    # # text2 = 'xyyxx'
    # text1 = 'abcabc'
    # text2 = 'abc abc'

    # val, trie = lcs(0, 0, text1, text2, dp={})

    # matches = []
    # flatten_trie_node(trie, [], matches)
    # # print(matches)

- miscellaneous

In [8]:
def token_matched_all_or_partial_but_in_full(root1, match1):
    rcnt, mcnt = {}, {}
    for r, m in zip(root1, match1):
        if r is not None:
            if r not in rcnt:
                rcnt[r] = 0
            if r not in mcnt:
                mcnt[r] = 0
                
            rcnt[r] += 1
            if m >= 0:
                mcnt[r] += 1

    perfect_matched = all(mcnt[k] == v for k,v in rcnt.items())
    partial_token_matched = all(mcnt[k]==0 or mcnt[k] == v for k,v in rcnt.items())
    return perfect_matched, partial_token_matched

In [9]:
def count_consecutive_initials(ii, i_not_set):
    conn = [0] * len(ii)
    for k,v in ii.items():
        if k not in i_not_set:
            conn[v] = 1

    cnt = 0
    for i in range(len(conn)):
        if conn[i] > 0 and (i == 0 or conn[i-1] == 0):
            cnt += 1

    return cnt

In [10]:
def is_two_texts_same(csum1, iroot1, match1, csum2, iroot2, match2):
    ii_remaining_1 = set(iroot1.keys()).intersection([i for i, (t,m) in enumerate(zip(csum1, match1)) if t > 0 and m < 0]) # if token and not assigned
    ii_remaining_2 = set(iroot2.keys()).intersection([i for i, (t,m) in enumerate(zip(csum2, match2)) if t > 0 and m < 0]) # if token and not assigned            

    cnt1 = count_consecutive_initials(iroot1, ii_remaining_1)
    cnt2 = count_consecutive_initials(iroot2, ii_remaining_2)

    is_same = (
        (len(ii_remaining_1) == 0 and (len(iroot1) <= len(iroot2) - len(ii_remaining_2))) or
        (len(ii_remaining_2) == 0 and (len(iroot2) <= len(iroot1) - len(ii_remaining_1)))
        ) and (cnt1 == 1 and cnt2 == 1)
    return is_same


# TEST

In [11]:
if __name__=='__main__':
    for text1, text2 in [
        # ('abc', 'abcabc'),
        # ('abc abc', 'abcabc'),
        # ('abc', 'aabbcc'),
        # ('abc abc', 'aabbcc'),
        # ('abc', 'aaaaaabc'),
        # ('xab', 'axb'),
        # ('axbc', 'abcx'),
        # ('abc abc', 'abcabcxa'),
        # ('aecxef ghi', 'aec xef ghi'),
        ('Alan Turing', 'Alan Mathison Turing'),
        ('Albert Einstein', 'Einstein, Albert'),
        ('Tom Edison', 'Thomas Edison'),
        ('Bomsoo Kim', 'Bom Soo Kim'),
        ('Bomsoo Kim', 'Kim, Bom-soo'),
        ('Bomsoo Kim', 'Bomsoo B. Kim'),
        ('Bomsoo Brad Kim', 'Bomsoo B. Kim'),
        ('Brad Pitt', 'Bomsoo Kim'),
        ('Tingting Wong', 'Ting Ting Wong'),
        ('Green Construction Corporation Ltd.', 'Green Construction Co., Limited'),
        ('International Inn Company Ltd', 'I. I. Co. Limited'),
        ('International I. Company Ltd', 'Int. Inn Co. Limited'),
        ('United States of America', 'U.S.A.'),
        ('United States of America', 'USA'),
        ('New York, NY ', 'NY, NY'),
        # ('Smith Bro., 123 Main Street, Unit 5, Scanton, A1B 2C3', 'Smith Brothers, 132 Main St. Unit 5, Scanton, A1B 2C4'),
        # ('Empire State Building, 20 W 34th St., New York, NY 10001 United States of America (Tel. 123-456-7890)', '20 WEST 34TH STREET, EMPIRE STATE BLD. 44TH FLOOR ROOM#2, NY, NY 10001-00000 U.S.A.'), # !!!!!!!!!!!!!!!! NY needs to be captured
        ('Yun Ying Jin', 'Yunying Jin'),
        ]:

        print(f'#####################################################################################')
        out = longest_common_token_subsequence(text1.lower(), text2.lower())

        csum1, root1, iroot1 = out['csum1'], out['root1'], out['iroot1']
        csum2, root2, iroot2 = out['csum2'], out['root2'], out['iroot2']

        for i, row in enumerate(out['matches']):
            match1, match2 = row['match1'], row['match2']

            log = log_matched_results(text1, text2, match1, match2)

            score1 = sum(n>=0 for n in match1) / sum(r is not None for r in root1)
            score2 = sum(n>=0 for n in match2) / sum(r is not None for r in root2)

            print(f"---- [{i+1}/{len(out['matches'])}] ---------------------------------------------------")
            print(log)
            print(f"score1 = {score1}, score2 = {score2}, avg(scores) = {0.5*(score1 + score2)}")

        # print(f'#####################################################################################')
        # out = longest_common_token_subsequence(text1.lower(), text2.lower())

        # csum1, root1, iroot1 = out['csum1'], out['root1'], out['iroot1']
        # csum2, root2, iroot2 = out['csum2'], out['root2'], out['iroot2']

        # for i, row in enumerate(out['matches']):
        #     match1, match2 = row['match1'], row['match2']
            
        #     log = log_matched_results(text1, text2, match1, match2)            

        #     #------------------------------------------------------------------------------------------------
        #     is_same = is_two_texts_same(csum1, iroot1, match1, csum2, iroot2, match2)
            
        #     perfect_matched1, partial_token_matched1 = token_matched_all_or_partial_but_in_full(root1, match1)
        #     perfect_matched2, partial_token_matched2 = token_matched_all_or_partial_but_in_full(root2, match2)

        #     print(f"---- [{i+1}/{len(out['matches'])}] ---------------------------------------------------")
        #     print(log)
        #     print(f'RESULT: {is_same or (partial_token_matched1 and partial_token_matched2 and (perfect_matched1 or perfect_matched2))}')


#####################################################################################
---- [1/1] ---------------------------------------------------
Alan Turing
^^^^ ^^^^^^
AAAA BBBBBB
Alan Mathison Turing
^^^^          ^^^^^^
AAAA          BBBBBB
score1 = 1.0, score2 = 0.5555555555555556, avg(scores) = 0.7777777777777778
#####################################################################################
---- [1/1] ---------------------------------------------------
Albert Einstein
^^^^^^ ^^^^^^^^
BBBBBB AAAAAAAA
Einstein, Albert
^^^^^^^^  ^^^^^^
AAAAAAAA  BBBBBB
score1 = 1.0, score2 = 1.0, avg(scores) = 1.0
#####################################################################################
---- [1/1] ---------------------------------------------------
Tom Edison
^^^ ^^^^^^
AAA BBBBBB
Thomas Edison
^ ^^   ^^^^^^
A AA   BBBBBB
score1 = 1.0, score2 = 0.75, avg(scores) = 0.875
#####################################################################################
---- [1/1] ------------